In [19]:
"""
手写实现sft监督微调
"""
from transformers import AutoTokenizer

In [20]:
# 分词器
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B-Base/")

In [21]:
# 模拟训练数据
message_list = [
    {'role':'user','content':'你是谁？'},
    {'role':'assistant','content':'qwen3'},
    {'role':'user','content':'你好啊'},
    {'role':'assistant','content':'hello'},
]

In [22]:
res = tokenizer.apply_chat_template(message_list)
print(res)
# attention_mask 用于多个 对话列表长度对齐时 在较短的对话中 用于区分对话内容和padding

{'input_ids': [151644, 872, 198, 105043, 100165, 11319, 151645, 198, 151644, 77091, 198, 80, 16948, 18, 151645, 198, 151644, 872, 198, 108386, 103924, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 14990, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [23]:
# 模拟预测数据
message_list2 = [
    {'role':'user','content':'你是谁？'},
    {'role':'assistant','content':'qwen3'},
    {'role':'user','content':'你好啊'},
]
# 实际使用过程中 tokenize=True 进行分词和替换id  此处为了方便查看
# enable_thinking = False 时自动补充一个空的 think标签对 从而防止模型思考
inference_res = tokenizer.apply_chat_template(message_list2,tokenize=False,add_generation_prompt=True,enable_thinking = False)
print(inference_res)

<|im_start|>user
你是谁？<|im_end|>
<|im_start|>assistant
qwen3<|im_end|>
<|im_start|>user
你好啊<|im_end|>
<|im_start|>assistant
<think>

</think>




In [24]:
from datasets import load_dataset
from typing import Dict

### 加载训练集函数

In [25]:
def get_train_data():
    """
    加载训练数据
    :return:
    """
    # 从下载路径加载数据
    # 数据内容参考官网: https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k/viewer/default/train_sft
    dataset = load_dataset('data/ultrachat_200k')

    # 取出训练部分
    train_dataset = dataset['train_sft']

    # 遍历每条数据并使用 tokenizer 编码
    result_list = []
    train_data_size = 200 # 训练集数量

    for i in range(train_data_size):
        message_list:list[Dict] = train_dataset[i]['messages']
        message_list.insert(0,{"role":"system",'content':'You are a helpful assistant.'})
        result_dict:dict = tokenizer.apply_chat_template(message_list)
        result_list.append(result_dict)
    return result_list

In [27]:
# 查看第一条数据
data_list = get_train_data()
data_list[0]

{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 9485, 11221, 3796, 311, 3772, 5980, 21386, 320, 78795, 220, 21, 13, 15, 44662, 10392, 2210, 220, 19, 13, 15, 44662, 4270, 44168, 220, 18, 13, 15, 10, 47175, 220, 17, 13, 15, 44662, 34906, 24078, 220, 20, 13, 15, 10, 568, 3555, 6912, 2319, 1079, 358, 1667, 5267, 1925, 697, 25326, 6816, 609, 50219, 25326, 14158, 11, 498, 646, 6707, 1473, 279, 14246, 2168, 315, 264, 1985, 389, 19548, 553, 27362, 825, 315, 279, 6912, 594, 5798, 3419, 5003, 4894, 7771, 11101, 6816, 609, 50219, 25326, 14158, 686, 1431, 3037, 279, 14246, 1985, 2168, 1101, 553, 68607, 916, 429, 1985, 2168, 28574, 624, 21468, 419, 4565, 3796, 311, 678, 14158, 315, 279, 6912, 476, 1101, 3151, 6174, 438, 10007, 304, 279, 1467, 3684, 30, 151645, 198, 151644, 77091, 198, 1986, 4565, 1172, 16790, 311, 11101, 6816, 323, 50219, 25326, 14158, 315, 279, 3772, 5980, 21386, 10007, 304, 279, 1467, 3684, 13, 151645, 198, 151644, 872, 198, 6713

### llm 回答掩码函数

In [28]:
from transformers import PreTrainedTokenizerFast
import torch
from typing import List

def create_answer_mask(input_ids,tokenizer:PreTrainedTokenizerFast):
    """
    创建answer mask，从input_ids当中找出assistant回答的部分，然后输出一个与input_ids相同shape的mask，
    后续将其与pad_mask进行逻辑与操作，得到最终的mask，用以计算损失
    """


    # 构建answer mask，输入的input_ids为批量 tokenize之后的数据，对于每一条数据，查找当中assistant回答的部分，将其设置为1

    # 1. 构造一个和input_ids相同shape的全0矩阵
    answer_mask = torch.zeros_like(input_ids)

    # 2. 遍历input_ids中的每一条数据，查找assistant回答的部分，将其设置为1
    eos_token_id = tokenizer.encode('<|im_end|>')[0]
    for idx,ids in enumerate(input_ids):
        # 获取到所有的eos_position
        eos_position:List = torch.where(ids == eos_token_id)[0].tolist()

        # 排除第一个eos_position: 第一个对应的是system prompt
        eos_position = eos_position[1:]
        # 解析获得user_ends和assistant_ends
        user_ends,assistant_ends = _parse_conversation_turns(eos_position)
        # 设置answer mask
        _set_answer_masks(answer_mask[idx],user_ends,assistant_ends)

    # 结果返回:
    return answer_mask

def _parse_conversation_turns(eos_positions:List[int]):
    """
    输入eos_positions，输出user所对应的end位置和assistant所对应的end位置。

    以下面的对话为例：
    <|im_start|>system
    You are a helpful assistant.<|im_end|>
    <|im_start|>user
    什么是习惯？<|im_end|>
    <|im_start|>assistant
    习惯是指在一定时间内重复执行的行为。<|im_end|>
    <|im_start|>user
    如何培养一个习惯<|im_end|>
    <|im_start|>assistant
    21天培养法，每天坚持xxx<|im_end|>

    假设第一个eos_token_id index为5，第二个为10，第三个为15，第四个为20，第五个为25，
    那么输入的eos_token_id为：[10,15,20,25]
    user_turns为从第一个开始取（具体索引位置需要加一，因为eos_token_id后面还有一个\n换行符），每隔一个取一次，assistant_turns为从第二个开始取，每隔一个取一次。

    输出结果为：
        user_turns:[11,21]
        assistant_ends:[16,26]
    """

    use_ends = [pos+1 for pos in eos_positions[::2]]
    assistant_ends = [pos+1 for pos in eos_positions[1::2]]

    return use_ends,assistant_ends

def _set_answer_masks(mask,user_ends,assistant_ends):
    """
    将mask当中，assistant回答的部分，设置为1（原地修改，不返回新的mask），其余部分保持为0

    以下面的对话为例：
    <|im_start|>system
    You are a helpful assistant.<|im_end|>
    <|im_start|>user
    什么是习惯？<|im_end|>
    <|im_start|>assistant
    习惯是指在一定时间内重复执行的行为。<|im_end|>
    <|im_start|>user
    如何培养一个习惯<|im_end|>
    <|im_start|>assistant
    21天培养法，每天坚持xxx<|im_end|>

    假设第一个eos_token_id index为5，第二个为10，第三个为15，第四个为20，第五个为25，
    那么user_turns:[11,21]，assistant_ends:[16,26]

    user_ends当中的索引指向的是<|im_end|>之后的\n的索引，
    assistant_ends当中的索引指向的是<|im_end|>之后的\n的索引，
    要想获取到assistant的回答的起始位置，就需要再跳过\n,<|im_start|>,assistant 这三个token，所以需要加3.
    要想获取到assistant的回答的结束位置，就需要往前跳一个<|im_end|>，所以需要减1.
    """
    num_user_turns = len(user_ends)
    num_assistant_turns = len(assistant_ends)
    if num_user_turns == num_assistant_turns:
        for user_end,assistant_end in zip(user_ends,assistant_ends):
            answer_start = user_end + 3
            answer_end = assistant_end - 1
            mask[answer_start:answer_end] = 1

    elif num_user_turns == num_assistant_turns + 1:
        for user_end,assistant_end in zip(user_ends[:-1],assistant_ends):
            answer_start = user_end + 3
            answer_end = assistant_end - 1
            mask[answer_start:answer_end] = 1

        # 处理最后一轮被截断的助手回答
        last_user_end = user_ends[-1]
        last_answer_start = last_user_end + 3
        mask[last_answer_start:] = 1

In [30]:
# 测试 answer mask
message_list = [
    {"role":"system","content":"你是一个智能助手"},
    {"role":"user","content":"你好"},
    {"role":"assistant","content":"你好呀！"},
    {"role":"user","content":"早上好"},
    {"role":"assistant","content":"hello"},
]
input_ids= tokenizer.apply_chat_template(message_list,tokenize=True)["input_ids"]

input_ids = torch.tensor([input_ids],dtype=torch.long)  # 套[] 为 batch_size 升维

assistant_answer_mask = create_answer_mask(input_ids=input_ids,tokenizer=tokenizer)
print(assistant_answer_mask)    # 标注为1 对应的内容是 llm的回答

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0]])


In [33]:
# 转为一维列表
input_id_list=input_ids.tolist()[0]
assistant_answer_mask_list = assistant_answer_mask.tolist()[0]

# repr(): 把字符串里所有不可见的控制字符、转义字符等打印出来
for input_id,is_assistant_answer in zip(input_id_list,assistant_answer_mask_list):
    print('当前的token_id为：',input_id, "解码之后的token是：",repr(tokenizer.decode(input_id)),"当前token是否为assistant_answer:",is_assistant_answer)


当前的token_id为： 151644 解码之后的token是： '<|im_start|>' 当前token是否为assistant_answer: 0
当前的token_id为： 8948 解码之后的token是： 'system' 当前token是否为assistant_answer: 0
当前的token_id为： 198 解码之后的token是： '\n' 当前token是否为assistant_answer: 0
当前的token_id为： 56568 解码之后的token是： '你' 当前token是否为assistant_answer: 0
当前的token_id为： 101909 解码之后的token是： '是一个' 当前token是否为assistant_answer: 0
当前的token_id为： 100168 解码之后的token是： '智能' 当前token是否为assistant_answer: 0
当前的token_id为： 110498 解码之后的token是： '助手' 当前token是否为assistant_answer: 0
当前的token_id为： 151645 解码之后的token是： '<|im_end|>' 当前token是否为assistant_answer: 0
当前的token_id为： 198 解码之后的token是： '\n' 当前token是否为assistant_answer: 0
当前的token_id为： 151644 解码之后的token是： '<|im_start|>' 当前token是否为assistant_answer: 0
当前的token_id为： 872 解码之后的token是： 'user' 当前token是否为assistant_answer: 0
当前的token_id为： 198 解码之后的token是： '\n' 当前token是否为assistant_answer: 0
当前的token_id为： 108386 解码之后的token是： '你好' 当前token是否为assistant_answer: 0
当前的token_id为： 151645 解码之后的token是： '<|im_end|>' 当前token是否为assistant_answer: 0
当前的toke

### 损失函数

In [39]:
# SFT损失计算函数
def compute_loss(output_logits, labels, assistant_answer_mask):
    """
    output_logits: 模型前向传播之后输出的结果    [shape batch_size, seq_len, vocab_size]
    labels: 真实的标签                         [batch_size, seq_len]
    assistant_answer_mask: 模型回答掩码
    """

    # 1、output_logits计算log softmax
    # log_probs: [batch_size, seq_len, vocab_size]
    log_probs = torch.log_softmax(output_logits, dim=-1)

    # 2、找到模型输出答案所对应的token的概率是多少
    result = torch.gather(
        log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    )

    result = result.squeeze(-1)

    # 最大化问题转为最小化 [batch_size,seq_len]
    negative_log_probs = result * (-1)

    # 利用模型回答掩码 0 or 1 去掉除模型回答之外的数值
    masked_negative_log_probs = negative_log_probs * assistant_answer_mask

    # 将序列当中所有token的对数概率加起来，除以总的token数量
    average_loss  = masked_negative_log_probs.sum() / assistant_answer_mask.sum()

    return average_loss

In [40]:
# torch.gather: 张量中 指定索引位置的额值
# 假设vocab_size=3
log_probs = torch.tensor(
    [[[0.1,0.3,0.6],[0.2,0.5,0.3],[0.4,0.4,0.2]]]   # 形状:[1, 3, 3] 含义: 1个批次，3个Token位置，每个Token有3个词的概率
)

labels = torch.tensor(      # 指定的索引
    [[2,0,1]]
)

result = torch.gather(
    log_probs,                      # 数据
    dim=-1,                         # 维度
    index=labels.unsqueeze(-1)      # 索引  unsqueeze()增加维度 labels形状: [1,3]  -> [1,3,1]: [[ [2],[0],[1] ]] 用于与 log_probs对齐
)
result

tensor([[[0.6000],
         [0.2000],
         [0.4000]]])

### 学习率调度器

In [36]:
import numpy as np
def cosine_scheduler_with_warmup(total_batch, warmup_ratio,lr,current_batch):
    """
    带预热的余弦衰减调度器
    :param total_batch: 总步数
    :param warmup_ratio:  预热比例
    :param lr:  学习率
    :param current_batch:   当前步数
    :return:
    """
    warmup_batch = total_batch * warmup_ratio      # 预热截止步数
    if current_batch < warmup_batch:    # 还在预热
        return current_batch * lr / warmup_batch    # 预热进度比例 * 最高学习率
    else:
        progress = (current_batch - warmup_batch) / (total_batch - warmup_batch)    # 下降进度比例
        # decay: 0.5 * (1 + cos(π*progress))
        # progress [0,1] π*progress [0,π] 对应cos函数  [0,π] 区间下降函数 对其进行平移伸缩 获取结果属于[0,1]
        decay =  0.5 * (1 + np.cos(np.pi*progress))
        current_lr = lr * decay
        return current_lr

### 训练函数

In [ ]:
from transformers import AutoModelForCausalLM
from dataclasses import dataclass
from torch.optim.adamw import AdamW
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import math

@dataclass
class SFTConfig:

    batch_size:int =4                           # 批次大小
    lr: float = 5e-5                            # 学习率
    warmup_ratio:float = 0.1                    # 预热步数比例
    log_dir:str = "logs/01_SFT_TRAIN"           # 日志保存路径
    log_batch:int = 100                         # 打印日志的间隔
    save_dir:str = "finetuned/01_SFT_TRAIN"     # 微调结果保存路径


def train(sft_confg:SFTConfig):
    # 1、获取模型，将模型置为train模式，放到cuda
    model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B-Base/")
    model.to("cuda")
    model.train()

    # 2、获取训练数据
    train_data_list:list = get_train_data()
    # 数学方式实现向上取整:  total_batch= (总数据量 + batch_size - 1) // batch_size
    # total_batch = (len(train_data_list) + sft_confg.batch_size -1) // sft_confg.batch_size
    # 库函数实现向上取整
    total_batch = math.ceil(len(train_data_list) / sft_confg.batch_size)

    # 3、构造优化器
    optimizer = AdamW(model.parameters(),lr=sft_confg.lr)

    # 5、添加日志
    # 5.1 使用tensorboard
    writer = SummaryWriter(log_dir=sft_confg.log_dir)
    # 5.2 使用tqdm
    progress_bar = tqdm(total=total_batch)


    losses_list = []
    for batch in range(total_batch):

        # 1、张量准备
        # 1.1 获取当前批次数据
        batch_train_data:list[dict] = train_data_list[batch * sft_confg.batch_size : (batch+1) * sft_confg.batch_size]
        # 1.2 数据padding
        batch_data_max_len = max([len(seq["input_ids"]) for seq in batch_train_data])   # 批次中的最大长度
        padded_seq_result = []
        for single_seq in batch_train_data:
            padding_lenth = batch_data_max_len - len(single_seq["input_ids"])
            # 在input_ids的最后，填充padding
            padded_seq_input_ids = single_seq["input_ids"].extend([tokenizer.pad_token_id] *padding_lenth ) # extend方式追加至最长长度
            padded_seq_result.append(padded_seq_input_ids)

        # 1.3 构造张量
        data_tensor = torch.tensor(padded_seq_result, dtype=torch.long).to("cuda")  # 转为张量
        full_mask = create_answer_mask(input_ids=data_tensor, tokenizer=tokenizer)  # ai回答掩码
        input_ids = data_tensor[:,:-1]  # [batch_size, seq_len]
        labels = data_tensor[:,1:]      # [batch_size, seq_len]
        assistant_mask = full_mask[:, 1:]   # 完整掩码截取

        # 2、模型前向传播
        output_logits = model(input_ids).logits
        # 3、损失计算
        loss = compute_loss(output_logits=output_logits,labels=labels,assistant_answer_mask=assistant_mask)
        # 将loss进行记录，后续用以打印 每 n 步的平均损失
        losses_list.append(loss.item())
        # 更新tqdm的进度条
        progress_bar.update(1)
        progress_bar.set_postfix(loss=f"current_loss:{loss.item():.4f}")


        # 4、反向传播
        loss.backward()
        # 5、参数更新
        # 5.1 更新学习率
        optimizer.param_groups[0]["lr"] = cosine_scheduler_with_warmup(total_batch,sft_confg.warmup_ratio,sft_confg.lr,batch)
        # 5.2 更新参数
        optimizer.step()
        # 5.3 梯度清空
        optimizer.zero_grad()

        # 是否应该记录
        should_log = batch % sft_confg.log_batch or batch == (total_batch-1)    # 100取整 或  最后一个
        if should_log:
            # 计算前面log_batch步的平均损失，然后写在tensorboard当中
            log_batch_loss = losses_list[-sft_confg.log_batch:]
            average_loss = sum(log_batch_loss) / len(log_batch_loss)
            writer.add_scalar("train_loss",scalar_value=average_loss, global_step=batch)


def save_model_tokenizer(model,tokenizer,sft_config:SFTConfig):
    """
    将训练之后的模型和tokenzier进行保存
    """
    model.save_pretrained(sft_config.save_dir)
    tokenizer.save_pretrained(sft_config.save_dir)
    print("当前模型和tokenizer保存完毕")